# 11. MCP (Model Context Protocol)

## 학습 목표

MCP(Model Context Protocol)로 외부 도구와 컨텍스트를 에이전트에 연결하는 방법을 알아봅니다.

이 노트북에서 다루는 내용:
- MCP의 개념과 아키텍처(서버/클라이언트/호스트)를 이해한다
- `langchain-mcp-adapters` 패키지로 MCP 서버에 연결한다
- `ChatOpenAI.bind_tools(mcp_tools)`로 에이전트와 MCP 도구를 통합한다
- Stdio와 SSE 전송 방식의 차이를 안다
- 다중 MCP 서버를 연결하는 방법을 익힌다

## 11.1 환경 설정

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("환경 준비 완료.")

환경 준비 완료.


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

Langfuse tracing ON — https://lf.ddok.ai


## 11.2 MCP 개념

**MCP(Model Context Protocol)**는 외부 도구와 컨텍스트를 **표준화된 방식**으로 LLM에 제공하기 위한 오픈 프로토콜입니다.

### 아키텍처 구성 요소

| 구성 요소 | 역할 | 예시 |
|----------|------|------|
| **MCP 서버** | 도구, 리소스, 프롬프트를 노출 | 파일 시스템 서버, DB 서버, API 래퍼 |
| **MCP 클라이언트** | 서버에 연결하여 도구를 가져옴 | `MultiServerMCPClient` |
| **호스트** | 클라이언트를 관리하고 LLM과 연결 | LangChain 에이전트, IDE |

### 핵심 리소스 타입

- **Tools**: 에이전트가 호출할 수 있는 실행 가능한 함수
- **Resources**: 파일, DB 레코드 등의 데이터 (LangChain Blob 객체로 변환)
- **Prompts**: 재사용 가능한 프롬프트 템플릿

### 왜 MCP인가?

MCP 이전에는 각 도구마다 연결 코드를 따로 작성해야 했습니다. MCP는 이를 **하나의 표준 프로토콜**로 통합합니다:
- 도구 제공자는 MCP 서버를 한 번만 구현하면 됩니다
- LLM 호스트는 MCP 클라이언트 하나로 모든 도구에 접근할 수 있습니다
- 생태계 전체에서 도구를 재사용할 수 있습니다

## 11.3 langchain-mcp-adapters 설치

LangChain에서 MCP를 사용하려면 `langchain-mcp-adapters` 패키지가 필요합니다.

In [ ]:
# MCP 어댑터 설치 명령어
print("MCP 어댑터 설치:")
print("  uv add langchain-mcp-adapters mcp")
print()
print("주요 컴포넌트:")
print("  - MultiServerMCPClient: 여러 MCP 서버를 관리하는 클라이언트")
print("  - load_mcp_tools(session): MCP 세션을 LangChain Tool로 변환")
print("  - FastMCP: 빠르게 MCP 서버를 만드는 서버 유틸리티")

## 11.4 Stdio 전송

**Stdio(Standard I/O)** 전송은 로컬 서브프로세스로 MCP 서버와 통신합니다. 개발 및 테스트 환경에 적합합니다.

In [ ]:
from pathlib import Path; import json, tempfile, sys
server_path = Path(tempfile.gettempdir()) / "lc_mcp_math_server.py"
server_path.write_text('from mcp.server.fastmcp import FastMCP\nmcp = FastMCP("math")\n@mcp.tool()\ndef add(a: int, b: int) -> int:\n    return a + b\nif __name__ == "__main__":\n    mcp.run(transport="stdio")')
stdio_config = {"math": {"transport": "stdio", "command": sys.executable, "args": [str(server_path)]}}
print("Stdio 전송 설정:"); print(json.dumps(stdio_config, indent=2))
print(f"\n서버 파일: {server_path}")

## 11.5 SSE/HTTP 전송

**HTTP(streamable-http)** 전송은 웹 기반 통신으로, 원격 MCP 서버에 연결할 때 사용합니다. 인증 헤더와 커스텀 설정을 지원합니다.

In [ ]:
# HTTP/streamable-http 전송 설정 예시
http_config = {
    "weather_server": {"transport": "streamable_http", "url": "https://weather-mcp.example.com/mcp", "headers": {"Authorization": "Bearer YOUR_API_KEY"}}
}
import json; print("HTTP 전송 설정:"); print(json.dumps(http_config, indent=2))
print("\n사용 패턴: client = MultiServerMCPClient(http_config) -> await client.get_tools()")

## 11.6 MCP 도구 로드 및 에이전트 통합

MCP 서버에서 가져온 도구를 LangChain 에이전트에 바인딩하는 패턴입니다.

In [ ]:
import asyncio
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
async def run_math_agent():
    client = MultiServerMCPClient(stdio_config)
    agent = create_agent(model=model, tools=await client.get_tools(), system_prompt="MCP math 도구를 사용할 수 있습니다.")
    return await agent.ainvoke({"messages": [{"role": "user", "content": "add 도구로 2와 3을 더해 주세요."}]}, config=lf_config)
result = asyncio.run(run_math_agent())
print(result["messages"][-1].content)

## 11.7 다중 MCP 서버 연결

`MultiServerMCPClient`는 이름 그대로 여러 MCP 서버를 동시에 관리할 수 있습니다.

In [ ]:
# 다중 MCP 서버 설정 예시
import json, sys
multi_server_config = {"math_server": {"transport": "stdio", "command": sys.executable, "args": [str(server_path)]}, "weather_server": {"transport": "streamable_http", "url": "https://weather-mcp.example.com/mcp"}, "database_server": {"transport": "stdio", "command": "npx", "args": ["-y", "@modelcontextprotocol/server-postgres"], "env": {"DATABASE_URL": "postgresql://..."}}}
print("다중 MCP 서버 설정:"); print(json.dumps(multi_server_config, indent=2, ensure_ascii=False))
print("\n사용 패턴: client = MultiServerMCPClient(multi_server_config) -> await client.get_tools()")
print("참고: 기본적으로 stateless — 각 도구 호출마다 새 세션 생성 후 정리")

## 11.8 도구 인터셉터

**Tool Interceptor**는 MCP 도구 호출을 가로채는 미들웨어입니다. 런타임 컨텍스트에 접근하거나 요청/응답을 수정하거나 재시도 로직을 구현할 수 있습니다.

### 인터셉터 사용 사례

| 사용 사례 | 설명 |
|----------|------|
| 인증 주입 | 런타임에 사용자별 토큰 전달 |
| 요청 수정 | 도구 호출 파라미터 변환 |
| 응답 필터링 | 민감한 정보 제거 |
| 재시도 로직 | 실패 시 자동 재시도 |
| 로깅 | 도구 호출 추적 |

In [ ]:
import asyncio
from langchain.agents import create_agent; from langchain_mcp_adapters.client import MultiServerMCPClient; from langchain_mcp_adapters.interceptors import ToolCallInterceptor
class LoggingInterceptor(ToolCallInterceptor):
    async def __call__(self, request, handler): print(f"Tool call: {request.name} @ {request.server_name}"); return await handler(request)
async def run_with_interceptor():
    client = MultiServerMCPClient(stdio_config, tool_interceptors=[LoggingInterceptor()])
    agent = create_agent(model=model, tools=await client.get_tools(), system_prompt="수학 도구를 사용하세요.")
    return await agent.ainvoke({"messages": [{"role": "user", "content": "add 도구로 7과 8을 더해 주세요."}]}, config=lf_config)
result = asyncio.run(run_with_interceptor()); print(result["messages"][-1].content)

## 11.9 커스텀 MCP 서버 작성

**FastMCP** 라이브러리를 쓰면 데코레이터로 간편하게 MCP 서버를 구축할 수 있습니다.

In [ ]:
from pathlib import Path; import asyncio, tempfile, sys
from langchain.agents import create_agent; from langchain_mcp_adapters.client import MultiServerMCPClient
fastmcp_path = Path(tempfile.gettempdir()) / "lc_fastmcp_server.py"
fastmcp_path.write_text('from mcp.server.fastmcp import FastMCP\nmcp = FastMCP("my-tools")\n@mcp.tool()\ndef add(a: int, b: int) -> int:\n    return a + b\n@mcp.tool()\ndef multiply(a: int, b: int) -> int:\n    return a * b\n@mcp.resource("config://app")\ndef get_config() -> str:\n    return \'{"version": "1.0", "debug": false}\'\nif __name__ == "__main__":\n    mcp.run(transport="stdio")')
async def run_custom_server():
    client = MultiServerMCPClient({"my_tools": {"transport": "stdio", "command": sys.executable, "args": [str(fastmcp_path)]}})
    agent = create_agent(model=model, tools=await client.get_tools(), system_prompt="곱셈 도구를 사용하세요.")
    return await agent.ainvoke({"messages": [{"role": "user", "content": "multiply 도구로 6과 7을 곱해 주세요."}]}, config=lf_config)
result = asyncio.run(run_custom_server()); print(result["messages"][-1].content)

## 11.10 요약

이 노트북에서 배운 내용:

| 주제 | 핵심 내용 |
|------|----------|
| **MCP 개념** | 외부 도구와 컨텍스트를 표준화된 방식으로 LLM에 제공하는 오픈 프로토콜입니다 |
| **Stdio 전송** | 로컬 서브프로세스를 통한 통신으로, 개발/테스트에 적합합니다 |
| **SSE/HTTP 전송** | 웹 기반 통신으로, 원격 서버 연결 및 인증을 지원합니다 |
| **에이전트 통합** | `client.get_tools()` → `create_agent(tools=mcp_tools)`로 연결합니다 |
| **다중 서버** | `MultiServerMCPClient`에 여러 서버를 설정하여 동시 관리합니다 |
| **인터셉터** | 도구 호출을 가로채 인증, 로깅, 수정 등의 미들웨어를 적용합니다 |
| **커스텀 서버** | `FastMCP`의 데코레이터로 간편하게 MCP 서버를 구축합니다 |

### 다음 단계
→ **[12. 프론트엔드 스트리밍](12_frontend_streaming.ipynb)**으로 진행하세요!

---
**참고 문서:**
- [MCP (Model Context Protocol)](../docs/langchain/16-mcp.md)